# Competition Model — Final Version

Key improvements over previous version:
- Richer features: cross-signal correlations, FFT, zero-crossing rate, dynamic stats
- **Subject-level signal baselines** (biggest lever for y_1)
- Leakage-free GroupKFold target encoding
- 4 models: LightGBM, XGBoost, CatBoost, HistGradientBoosting
- QuantileRegressor stacking (directly minimises MAE)
- Prediction clipping

## 1. Imports

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import QuantileRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [6]:
train_raw = pd.read_csv('train_competition_2026.csv')
test_raw  = pd.read_csv('test_no_outcome.csv')

train_raw['time'] = pd.to_datetime(train_raw['time'])
test_raw['time']  = pd.to_datetime(test_raw['time'])

print(f'Train raw: {train_raw.shape}')
print(f'Test raw:  {test_raw.shape}')

Train raw: (432600, 18)
Test raw:  (103500, 16)


## 3. Feature Engineering

In [7]:
def engineer_features(df):
    t_cols   = [f't_{i}' for i in range(5)]
    num_cols = ['num_0', 'num_1', 'num_2']
    cat_cols = [f'cat_{i}' for i in range(5)]

    df = df.sort_values(['obs', 'time']).copy()

    # --- Step 1: Basic aggregations ---
    agg_dict = {}
    for c in num_cols + cat_cols:
        agg_dict[c] = 'first'
    agg_dict['sub_id'] = 'first'
    for c in t_cols:
        agg_dict[c] = ['mean', 'std', 'min', 'max', 'first', 'last', 'median']

    grouped = df.groupby('obs').agg(agg_dict)
    grouped.columns = ['_'.join(col).strip('_') for col in grouped.columns]
    grouped = grouped.reset_index()

    # --- Step 2: Derived stats ---
    for c in t_cols:
        grouped[f'{c}_slope'] = grouped[f'{c}_last'] - grouped[f'{c}_first']
        grouped[f'{c}_range'] = grouped[f'{c}_max']  - grouped[f'{c}_min']
        grouped[f'{c}_cv']    = grouped[f'{c}_std']  / (grouped[f'{c}_mean'].abs() + 1e-8)

    # --- Step 3: Quantiles ---
    quantile_feats = df.groupby('obs')[t_cols].quantile([0.1, 0.25, 0.75, 0.9])
    quantile_feats = quantile_feats.unstack(level=-1)
    quantile_feats.columns = [f'{c}_q{int(q*100)}' for c, q in quantile_feats.columns]
    grouped = grouped.merge(quantile_feats.reset_index(), on='obs')

    # --- Step 4: Skewness and kurtosis ---
    skew_feats = df.groupby('obs')[t_cols].skew()
    skew_feats.columns = [f'{c}_skew' for c in t_cols]
    grouped = grouped.merge(skew_feats.reset_index(), on='obs')

    kurt_feats = df.groupby('obs')[t_cols].apply(lambda x: x.kurtosis())
    kurt_feats.columns = [f'{c}_kurt' for c in t_cols]
    grouped = grouped.merge(kurt_feats.reset_index(), on='obs')

    # --- Step 5: Signal interaction features ---
    grouped['t0_minus_t1'] = grouped['t_0_mean'] - grouped['t_1_mean']
    grouped['t2_minus_t3'] = grouped['t_2_mean'] - grouped['t_3_mean']
    grouped['t_mean_all']  = grouped[[f't_{i}_mean' for i in range(5)]].mean(axis=1)

    # Signal ratios
    for c1, c2 in combinations(t_cols, 2):
        grouped[f'{c1}_{c2}_ratio'] = grouped[f'{c1}_mean'] / (grouped[f'{c2}_mean'].abs() + 1e-8)

    # --- Step 6: Static numeric interactions ---
    grouped['num0_times_num1'] = grouped['num_0_first'] * grouped['num_1_first']
    grouped['num0_times_num2'] = grouped['num_0_first'] * grouped['num_2_first']

    # --- Step 7: Time features ---
    time_feats = df.groupby('obs')['time'].first()
    grouped['hour']       = pd.to_datetime(time_feats.values).hour
    grouped['dayofweek']  = pd.to_datetime(time_feats.values).dayofweek
    grouped['is_weekend'] = (grouped['dayofweek'] >= 5).astype(int)

    # --- Step 8: Dynamic / time-aware stats ---
    df['_tsec'] = (df['time'] - df.groupby('obs')['time'].transform('first')).dt.total_seconds()

    meta = df.groupby('obs').agg(
        n_points=('time', 'size'),
        duration_seconds=('_tsec', 'max')
    ).reset_index()
    grouped = grouped.merge(meta, on='obs', how='left')

    def dyn_stats(block):
        out = {}
        t = block['_tsec'].values.astype(float)
        n = len(t)
        if n < 2:
            for c in t_cols:
                x = block[c].values.astype(float)
                out[f'{c}_mean_abs_diff']   = 0.0
                out[f'{c}_last_minus_mean'] = float(x[-1] - np.mean(x)) if n == 1 else 0.0
                out[f'{c}_slope_lr']        = 0.0
            return pd.Series(out)
        tc    = t - t.mean()
        denom = float(np.sum(tc**2)) + 1e-12
        for c in t_cols:
            x  = block[c].values.astype(float)
            xc = x - x.mean()
            out[f'{c}_mean_abs_diff']   = float(np.mean(np.abs(np.diff(x))))
            out[f'{c}_last_minus_mean'] = float(x[-1] - np.mean(x))
            out[f'{c}_slope_lr']        = float(np.sum(tc * xc) / denom)
        return pd.Series(out)

    dyn = df.groupby('obs', sort=False).apply(dyn_stats).reset_index()
    grouped = grouped.merge(dyn, on='obs', how='left')
    df.drop(columns=['_tsec'], inplace=True)

    # --- Step 9: Subject obs count ---
    sub_counts = df.groupby('sub_id')['obs'].nunique().reset_index()
    sub_counts.columns = ['sub_id', 'sub_obs_count']
    grouped = grouped.merge(sub_counts, left_on='sub_id_first', right_on='sub_id', how='left')
    grouped = grouped.drop(columns=['sub_id'])

    # --- Step 10: Cross-signal Pearson correlations ---
    def cross_correlations(block):
        out  = {}
        vals = {c: block[c].values.astype(float) for c in t_cols}
        for c1, c2 in combinations(t_cols, 2):
            v1, v2 = vals[c1], vals[c2]
            denom  = (np.std(v1) * np.std(v2)) + 1e-12
            out[f'{c1}_{c2}_corr'] = float(np.corrcoef(v1, v2)[0, 1]) if denom > 1e-10 else 0.0
        return pd.Series(out)

    corr_feats = df.groupby('obs', sort=False).apply(cross_correlations).reset_index()
    grouped = grouped.merge(corr_feats, on='obs', how='left')

    # --- Step 11: FFT features ---
    def fft_feats(block):
        out = {}
        for c in t_cols:
            x   = block[c].values.astype(float)
            fft = np.abs(np.fft.rfft(x - x.mean()))
            out[f'{c}_fft_energy']    = float(np.sum(fft**2))
            out[f'{c}_fft_peak_freq'] = float(np.argmax(fft[1:]) + 1) if len(fft) > 1 else 0.0
        return pd.Series(out)

    fft_df = df.groupby('obs', sort=False).apply(fft_feats).reset_index()
    grouped = grouped.merge(fft_df, on='obs', how='left')

    # --- Step 12: Zero-crossing rate ---
    def zero_crossings(block):
        out = {}
        for c in t_cols:
            x = block[c].values.astype(float) - block[c].mean()
            out[f'{c}_zcr'] = float(np.sum(np.diff(np.sign(x)) != 0))
        return pd.Series(out)

    zcr_df = df.groupby('obs', sort=False).apply(zero_crossings).reset_index()
    grouped = grouped.merge(zcr_df, on='obs', how='left')

    return grouped

In [8]:
train_agg = engineer_features(train_raw)
test_agg  = engineer_features(test_raw)

targets   = train_raw.groupby('obs')[['y_1', 'y_2']].first().reset_index()
train_agg = train_agg.merge(targets, on='obs')

print(f'Train: {train_agg.shape}')
print(f'Test:  {test_agg.shape}')

Train: (14420, 153)
Test:  (3450, 151)


## 4. Subject-Level Signal Baselines

For each subject, compute their average signal level across ALL their observations.
This gives the model a 'baseline' for each subject — key for predicting y_1.

In [9]:
t_cols = [f't_{i}' for i in range(5)]

# Compute subject-level stats from train_raw only (no leakage)
sub_signal_stats = train_raw.groupby('sub_id')[t_cols].agg(['mean', 'std']).reset_index()
sub_signal_stats.columns = ['sub_id'] + [
    f'sub_{c}_{s}' for c in t_cols for s in ['mean', 'std']
]

# Merge into train and test
train_agg = train_agg.merge(sub_signal_stats, left_on='sub_id_first', right_on='sub_id', how='left').drop(columns='sub_id')
test_agg  = test_agg.merge(sub_signal_stats,  left_on='sub_id_first', right_on='sub_id', how='left').drop(columns='sub_id')

# Fill unseen test subjects with global mean
for c in t_cols:
    for s in ['mean', 'std']:
        col = f'sub_{c}_{s}'
        global_val = train_agg[col].mean()
        train_agg[col] = train_agg[col].fillna(global_val)
        test_agg[col]  = test_agg[col].fillna(global_val)

# How far is THIS observation from subject's baseline?
for c in t_cols:
    train_agg[f'{c}_vs_sub_mean'] = train_agg[f'{c}_mean'] - train_agg[f'sub_{c}_mean']
    test_agg[f'{c}_vs_sub_mean']  = test_agg[f'{c}_mean']  - test_agg[f'sub_{c}_mean']

print(f'Train after subject features: {train_agg.shape}')
print(f'Test  after subject features: {test_agg.shape}')

Train after subject features: (14420, 168)
Test  after subject features: (3450, 166)


## 5. Prepare Features + Label Encoding

In [10]:
drop_cols = ['obs', 'sub_id_first', 'y_1', 'y_2']

cat_features_raw = [f'cat_{i}_first' for i in range(5)]
label_encoders   = {}
for c in cat_features_raw:
    le       = LabelEncoder()
    all_vals = pd.concat([train_agg[c], test_agg[c]]).astype(str)
    le.fit(all_vals)
    train_agg[c] = le.transform(train_agg[c].astype(str))
    test_agg[c]  = le.transform(test_agg[c].astype(str))
    label_encoders[c] = le

feature_cols = [c for c in train_agg.columns if c not in drop_cols]

X_all  = train_agg[feature_cols].copy()
y_all  = train_agg[['y_1', 'y_2']].copy()
groups = train_agg['sub_id_first'].values
X_test = test_agg[feature_cols].copy()

X_all  = X_all.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

print(f'Feature count: {len(feature_cols)}')
print(f'Train obs: {len(X_all)}, Test obs: {len(X_test)}')

Feature count: 164
Train obs: 14420, Test obs: 3450


## 6. Leakage-Free Target Encoding on sub_id

In [11]:
def target_encode_subject(train_df, test_df, target_col, group_col='sub_id_first', n_splits=5):
    """GroupKFold target encoding — same subject never in both train and val."""
    train_df = train_df.copy()
    test_df  = test_df.copy()
    col_name = f'{group_col}_te_{target_col}'

    train_df[col_name] = np.nan
    gkf        = GroupKFold(n_splits=n_splits)
    sub_groups = train_df[group_col].values

    for tr_idx, va_idx in gkf.split(train_df, groups=sub_groups):
        means = train_df.iloc[tr_idx].groupby(group_col)[target_col].mean()
        train_df.iloc[va_idx, train_df.columns.get_loc(col_name)] = \
            train_df.iloc[va_idx][group_col].map(means)

    global_mean        = train_df[target_col].mean()
    train_df[col_name] = train_df[col_name].fillna(global_mean)
    overall            = train_df.groupby(group_col)[target_col].mean()
    test_df[col_name]  = test_df[group_col].map(overall).fillna(global_mean)

    return train_df, test_df

for t in ['y_1', 'y_2']:
    train_agg, test_agg = target_encode_subject(train_agg, test_agg, t)

# Refresh feature cols and arrays after adding target encoding
feature_cols = [c for c in train_agg.columns if c not in drop_cols]
X_all  = train_agg[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
X_test = test_agg[feature_cols].replace([np.inf, -np.inf], np.nan).copy()

print(f'Feature count after target encoding: {len(feature_cols)}')

Feature count after target encoding: 166


## 7. Train Models (LightGBM + XGBoost + CatBoost + HistGBM)

In [12]:
N_FOLDS    = 5
SEEDS      = [42, 123, 2026]
TARGETS    = ['y_1', 'y_2']
MODEL_KEYS = ['lgb', 'xgb', 'cat', 'hgb']
gkf        = GroupKFold(n_splits=N_FOLDS)

n_train = len(X_all)
n_test  = len(X_test)
n_seeds = len(SEEDS)

oof_preds  = {m: {t: np.zeros(n_train) for t in TARGETS} for m in MODEL_KEYS}
test_preds = {m: {t: np.zeros(n_test)  for t in TARGETS} for m in MODEL_KEYS}

for seed in SEEDS:
    print(f'SEED {seed}')
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups)):
        print(f'  Fold {fold+1}/{N_FOLDS}')
        X_tr, X_va = X_all.iloc[tr_idx], X_all.iloc[va_idx]
        y_tr, y_va = y_all.iloc[tr_idx], y_all.iloc[va_idx]

        for target in TARGETS:

            # --- LightGBM ---
            m_lgb = lgb.LGBMRegressor(
                objective='mae', metric='mae', verbosity=-1,
                n_estimators=5000, learning_rate=0.02, num_leaves=95,
                min_child_samples=30, subsample=0.75, colsample_bytree=0.55,
                reg_alpha=0.3, reg_lambda=3.0, random_state=seed,
                n_jobs=-1, subsample_freq=1,
            )
            m_lgb.fit(
                X_tr, y_tr[target],
                eval_set=[(X_va, y_va[target])],
                callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)]
            )
            oof_preds['lgb'][target][va_idx] += m_lgb.predict(X_va)
            test_preds['lgb'][target]        += m_lgb.predict(X_test) / (N_FOLDS * n_seeds)

            # --- XGBoost ---
            m_xgb = xgb.XGBRegressor(
                objective='reg:absoluteerror', eval_metric='mae',
                n_estimators=5000, learning_rate=0.02, max_depth=7,
                min_child_weight=30, subsample=0.75, colsample_bytree=0.55,
                reg_alpha=0.3, reg_lambda=3.0, gamma=0.05,
                early_stopping_rounds=150,
                random_state=seed, n_jobs=-1, verbosity=0, tree_method='hist',
            )
            m_xgb.fit(X_tr, y_tr[target], eval_set=[(X_va, y_va[target])], verbose=False)
            oof_preds['xgb'][target][va_idx] += m_xgb.predict(X_va)
            test_preds['xgb'][target]        += m_xgb.predict(X_test) / (N_FOLDS * n_seeds)

            # --- CatBoost ---
            m_cat = CatBoostRegressor(
                loss_function='MAE', iterations=5000, learning_rate=0.02, depth=7,
                l2_leaf_reg=3.0, min_data_in_leaf=30, random_seed=seed,
                verbose=0, subsample=0.75, bagging_temperature=0.5,
            )
            m_cat.fit(X_tr, y_tr[target], eval_set=(X_va, y_va[target]), early_stopping_rounds=150)
            oof_preds['cat'][target][va_idx] += m_cat.predict(X_va)
            test_preds['cat'][target]        += m_cat.predict(X_test) / (N_FOLDS * n_seeds)

            # --- HistGradientBoosting ---
            m_hgb = HistGradientBoostingRegressor(
                loss='absolute_error', max_iter=1000, learning_rate=0.02,
                max_leaf_nodes=63, min_samples_leaf=30, l2_regularization=3.0,
                random_state=seed, early_stopping=False,
            )
            m_hgb.fit(X_tr.fillna(-9999), y_tr[target])
            oof_preds['hgb'][target][va_idx] += m_hgb.predict(X_va.fillna(-9999))
            test_preds['hgb'][target]        += m_hgb.predict(X_test.fillna(-9999)) / (N_FOLDS * n_seeds)

# Average OOF across seeds
for model in MODEL_KEYS:
    for target in TARGETS:
        oof_preds[model][target] /= n_seeds

SEED 42
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
SEED 123
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
SEED 2026
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5


## 8. Evaluate Individual Models

In [13]:
y1_true = y_all['y_1'].values
y2_true = y_all['y_2'].values

print('Individual Model OOF Scores')
print('-' * 55)
for name, key in [('LightGBM', 'lgb'), ('XGBoost', 'xgb'), ('CatBoost', 'cat'), ('HistGBM', 'hgb')]:
    m1 = mean_absolute_error(y1_true, oof_preds[key]['y_1'])
    m2 = mean_absolute_error(y2_true, oof_preds[key]['y_2'])
    print(f'  {name:12s} → y1: {m1:.4f}, y2: {m2:.4f}, avg: {(m1+m2)/2:.4f}')

Individual Model OOF Scores
-------------------------------------------------------
  LightGBM     → y1: 4.7056, y2: 3.3181, avg: 4.0118
  XGBoost      → y1: 4.7076, y2: 3.3166, avg: 4.0121
  CatBoost     → y1: 4.6645, y2: 3.2833, avg: 3.9739
  HistGBM      → y1: 4.7091, y2: 3.3341, avg: 4.0216


In [14]:
# Load NN predictions (run AFTER training, BEFORE stacking)
oof_preds['nn']  = {'y_1': np.load('nn_oof_y1.npy'),  'y_2': np.load('nn_oof_y2.npy')}
test_preds['nn'] = {'y_1': np.load('nn_test_y1.npy'), 'y_2': np.load('nn_test_y2.npy')}

for t in ['y_1', 'y_2']:
    mae = mean_absolute_error(y_all[t].values, oof_preds['nn'][t])
    print(f'NN standalone {t} MAE: {mae:.4f}')

# Now stack all 5 models
ALL_KEYS = ['lgb', 'xgb', 'cat', 'hgb', 'nn']

NN standalone y_1 MAE: 4.8519
NN standalone y_2 MAE: 3.4115


## 9. Stack with QuantileRegressor

Uses OOF predictions as meta-features. QuantileRegressor at q=0.5 directly minimises MAE.

In [16]:
def stack_and_evaluate(oof_preds, test_preds, y_true, target, model_keys):
    meta_tr = np.column_stack([oof_preds[m][target]  for m in model_keys])
    meta_te = np.column_stack([test_preds[m][target] for m in model_keys])

    meta = QuantileRegressor(quantile=0.5, alpha=0.01, solver='highs')
    meta.fit(meta_tr, y_true)

    oof_stack  = meta.predict(meta_tr)
    test_stack = meta.predict(meta_te)

    mae = mean_absolute_error(y_true, oof_stack)
    print(f'  Stacked {target}: MAE = {mae:.4f}  |  weights: {np.round(meta.coef_, 3)}')
    return oof_stack, test_stack

print('Stacked Ensemble Results')
print('-' * 55)
oof_y1, final_y1 = stack_and_evaluate(oof_preds, test_preds, y1_true, 'y_1', ALL_KEYS)
oof_y2, final_y2 = stack_and_evaluate(oof_preds, test_preds, y2_true, 'y_2', ALL_KEYS)

print(f'\n  Final Avg MAE: {(mean_absolute_error(y1_true, oof_y1) + mean_absolute_error(y2_true, oof_y2))/2:.4f}')

Stacked Ensemble Results
-------------------------------------------------------
  Stacked y_1: MAE = 4.6409  |  weights: [0.    0.    0.538 0.307 0.173]
  Stacked y_2: MAE = 3.2626  |  weights: [0.    0.019 0.554 0.215 0.22 ]

  Final Avg MAE: 3.9518


## 10. Generate Submission

In [18]:
# Clip predictions to training range (avoids wild extrapolation)
final_y1 = np.clip(final_y1, y_all['y_1'].min(), y_all['y_1'].max())
final_y2 = np.clip(final_y2, y_all['y_2'].min(), y_all['y_2'].max())

submission = pd.DataFrame({'obs': test_agg['obs'], 'y_1': final_y1, 'y_2': final_y2})
submission.to_csv('nn_sample_submission.csv', index=False)

print(f'Saved {len(submission)} rows to sample_submission.csv')
submission.head(10)

Saved 3450 rows to sample_submission.csv


,obs,y_1,y_2
0,18,41.778503,104.132375
1,19,34.219758,99.574998
2,20,36.611674,96.008967
3,21,35.942596,94.186099
4,22,35.969859,94.175298
5,23,33.331057,57.098989
6,24,42.872394,58.275296
7,25,36.093536,54.360255
8,26,39.151670,49.418149
9,27,35.210226,49.539832


In [21]:
sub_weighted = pd.DataFrame({
    'obs': test_agg['obs'],
    'y_1': np.clip(
        0.7*test_preds['cat']['y_1'] + 0.2*test_preds['hgb']['y_1'] + 0.1*test_preds['nn']['y_1'],
        y_all['y_1'].min(), y_all['y_1'].max()),
    'y_2': np.clip(
        0.7*test_preds['cat']['y_2'] + 0.2*test_preds['hgb']['y_2'] + 0.1*test_preds['nn']['y_2'],
        y_all['y_2'].min(), y_all['y_2'].max())
})
sub_weighted.to_csv('submission_weighted.csv', index=False)